# Análisis de Componentes Principales (PCA)
## Proyecto Integrador — Minería de Datos I

---

### Objetivo de este notebook

Aplicar reducción de dimensionalidad mediante PCA sobre las variables numéricas del dataset.
Se documentarán las variables utilizadas, el proceso de escalamiento, la varianza explicada y la interpretación de los componentes obtenidos.


## 1. Setup Inicial y Selección de Variables
Importamos las librerías necesarias para el preprocesamiento y PCA. 
Solo seleccionaremos las variables estrictamente numéricas y continuas para este análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Cargar dataset limpio
df = pd.read_csv('../data/processed/streaming_users_clean.csv')

# Seleccionamos variables numéricas para PCA
# (Excluimos user_id porque es un identificador, no una variable analítica)
features = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
X = df[features]

print(f"Variables seleccionadas para PCA: {features}")
X.head()

## 2. Escalamiento de Datos

**Justificación:** PCA es un algoritmo basado en varianza y distancias. Si no escalamos los datos, la variable `monthly_watch_time_mins` (cuyos valores superan los 4000) dominaría completamente sobre `customer_support_tickets` (que varía entre 0 y 20). 
Aplicamos **StandardScaler** para estandarizar las variables, logrando que todas tengan media 0 y varianza 1, pesando por igual en el modelo.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificamos que la media sea aprox 0 y desviación estándar 1
df_scaled = pd.DataFrame(X_scaled, columns=features)
df_scaled.describe().round(2).loc[['mean', 'std']]

## 3. Aplicación de PCA y Varianza Explicada

Aplicamos PCA sobre los datos escalados. Nuestro objetivo inicial es ver cuántos componentes necesitamos para explicar al menos el 80% o 90% de la varianza original.

In [ ]:
# Instanciamos y entrenamos PCA con todas las componentes posibles (3 en este caso)
pca = PCA()
pca.fit(X_scaled)

# Extraemos la varianza explicada y la varianza acumulada
varianza_explicada = pca.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

for i, var in enumerate(varianza_explicada):
    print(f"Componente {i+1}: Explica el {var*100:.2f}% de la varianza (Acumulado: {varianza_acumulada[i]*100:.2f}%)")

### Gráfico 1: Varianza Explicada (Scree Plot)
Visualizamos la varianza para tomar la decisión de cuántos componentes retener.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Gráfico de barras para la varianza individual
ax.bar(range(1, len(varianza_explicada)+1), varianza_explicada, alpha=0.7, align='center',
       label='Varianza individual', color='skyblue')

# Línea para la varianza acumulada
ax.step(range(1, len(varianza_acumulada)+1), varianza_acumulada, where='mid',
        label='Varianza acumulada', color='red', marker='o')

ax.set_ylabel('Ratio de Varianza Explicada')
ax.set_xlabel('Componentes Principales')
ax.set_title('Gráfico de Sedimentación (Scree Plot)')
ax.set_xticks(range(1, len(varianza_explicada)+1))
ax.legend(loc='best')
plt.axhline(y=0.9, color='k', linestyle='--', alpha=0.3) # Línea de referencia del 90%
plt.tight_layout()
plt.show()

**Interpretación del Scree Plot:** 
Como tenemos 3 variables originales, el modelo nos devuelve 3 componentes. Vemos que para retener una cantidad significativa de varianza (~90%), tendríamos que usar las 3 componentes. No existe un 'codo' pronunciado que nos permita reducir la dimensionalidad drásticamente sin perder información. Esto ocurre cuando las variables originales tienen una correlación muy baja entre sí (ortogonales).

## 4. Visualización de los Componentes Principales

Para entender visualmente qué agrupa PCA, reducimos el dataset a las 2 primeras componentes principales (PC1 y PC2) y graficamos los usuarios, coloreados por plan de suscripción.

In [ ]:
# Transformamos los datos a los 2 primeros componentes
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

# Añadimos los componentes al dataframe original para graficar
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

# Gráfico 2: Dispersión de Componentes (Biplot simple)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='PC1', y='PC2', hue='subscription_plan', 
                palette='Set1', alpha=0.5, ax=ax)

ax.set_title('Proyección de Usuarios en Componentes Principales (PC1 vs PC2)')
ax.set_xlabel(f'Componente 1 ({varianza_explicada[0]*100:.1f}%)')
ax.set_ylabel(f'Componente 2 ({varianza_explicada[1]*100:.1f}%)')
plt.legend(title='Plan de Suscripción')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Conclusiones del Análisis PCA

**1. Dimensionalidad y Reducción:**
Debido a la nula correlación natural entre la edad, los minutos de consumo y los tickets de soporte, la varianza del dataset está uniformemente distribuida. PCA no logró reducir el dataset de 3 a 2 dimensiones reteniendo el clásico 90% de la varianza (solo logra capturar aprox. un 66% entre PC1 y PC2).

**2. Análisis de Clústeres (Gráfico de Componentes):**
Al proyectar a los usuarios sobre los primeros dos componentes principales, observamos una gran nube de dispersión cuadrada y homogénea. Los usuarios de distintos planes de suscripción (colores) están perfectamente superpuestos.

**Conclusión Final:**
La reducción de dimensionalidad reafirma los hallazgos del EDA: el comportamiento de los clientes de esta plataforma de streaming es altamente homogéneo. No existen "tipos" o arquetipos de clientes matemáticamente separables; un usuario Premium puede tener exactamente el mismo patrón de consumo, edad y quejas que un usuario Básico.